In [ ]:
import pandas as pd
import re
import seaborn as sns

In [ ]:
def read_results_file(fname):
  # Read and parse the file into a DataFrame
  data = []
  with open(file_path, 'r') as f:
      for line in f:
          # Split the line into key-value pairs and remove the trailing comma
          pairs = [item.strip().rstrip(',') for item in line.split(',')]
          # Convert the pairs into a dictionary
          row = {key_value.split('=')[0]: key_value.split('=')[1] for key_value in pairs if '=' in key_value}
          data.append(row)
  
  # Create the DataFrame
  df = pd.DataFrame(data)
  
  # Convert numeric columns to the appropriate type
  df = df.apply(pd.to_numeric, errors='ignore')
  return df

### Number of Background Clips

In [ ]:
# File path
file_path = "experiment_data/num_clips_test_results.dat"
df = read_results_file(file_path)

num_clips_vals = [int(re.search(r'_(\d+)clips', s).group(1)) for s in df['train_set']]
df.insert(2, 'num_clips', value=num_clips_vals)
df

In [ ]:
plt.figure(figsize=(12,6))
hAx1 = plt.subplot(1,2,1)
sns.scatterplot(data=df, ax=hAx1, x='false_detections', y='false_rejections', hue='num_clips')
hAx1.set_title(f"False Detections vs False Rejections w/ Threshold for Precision=0.95")
hAx2 = plt.subplot(1,2,2)
sns.scatterplot(data=df, ax=hAx2, x='fp_th0p99', y='fn_th0p99', hue='num_clips')
hAx1.set_title(f"False Detections vs False Rejections w/ Threshold for Precision=0.95")
hAx1.set_title(f"False Detections vs False Rejections w/ Threshold fixed at 0.99")

# hAx.set_xlim([0,40])
plt.tight_layout()

The bottom line takeaway from these results is that, at least at the current values for other parameters, there is no clear pattern of improved performance for different numbers of background clips in the range [50,100,200].

### Minimum SNR Level

In [ ]:
# File path
file_path = "experiment_data/snr_results_5jan2024a.dat"
df = read_results_file(file_path)

df.insert(2, 'num_clips', value=50)
snr_levels = [0.1, 0.1, 0.1, 0.2, 0.2, 0.2, 0.5, 0.5, 0.5, 1.0, 1.0, 1.0]
df.insert(3, 'snr_val', value=snr_levels)

df

In [ ]:
plt.figure(figsize=(8,5))
hAx1 = plt.subplot(1,2,1)
sns.scatterplot(data=df, ax=hAx1, x='fp_th0p95', y='fn_th0p95', hue='snr_val')
hAx1.set_title(f"Threshold fixed at 0.95")
hAx2 = plt.subplot(1,2,2)
sns.scatterplot(data=df, ax=hAx2, x='fp_th0p99', y='fn_th0p99', hue='snr_val')
hAx2.set_title(f"Threshold fixed at 0.99")

# hAx.set_xlim([0,40])
plt.tight_layout()

In [ ]:
# hAx1 = plt.gca()
sns.scatterplot(data=df, x='loss', y='sparse1p0', hue='snr_val')


The effect of SNR value during training here is not clear.  There's a slight pattern of lower SNRs giving fewer false positives, but it's not a strong one.

In this experiment, it looks like there is a very small effect wherein training on lower SNR values results in lower sparsity, though there is no reason to expect any significant sparsity here, because there is no L2 or L1 regularization, and the effect is very small -- all runs had sparsity at the 1% level in the range of 3.7-4.4%.

### Model Size Test 

In [ ]:
# File path
file_path = "experiment_data/model_test_s16_results.dat"
df16 = read_results_file(file_path)
df16["train_set"] = "sww_stride16.0_50clips_snr0.1_t1"
df16.insert(2, 'num_clips', value=50)
df16['num_params'][3:]=[44299, 44299, 44299, 119883, 119883, 119883, 376843, 376843, 376843, 522763, 522763, 522763]

file_path = "experiment_data/model_test_s32_results.dat"
df32 = read_results_file(file_path)
df32["train_set"] = "sww_stride32.0_50clips_snr0.1_t1"
df32.insert(2, 'num_clips', value=50)
num_params_s32 = [46875, 46875, 46875, 44235, 44235, 44235, 119755, 119755, 119755, 376587, 376587, 376587, 522507, 522507, 522507]
df32.insert(19, 'num_params', value=num_params_s32)

df = pd.concat([df16, df32], ignore_index=True)
df


In [ ]:
sns.scatterplot

In [ ]:
hAx2 = plt.gca()
sns.scatterplot(data=df, ax=hAx2, x='fp_th0p99', y='fn_th0p99', 
                hue='num_params', style='win_stride', markers={16.0: "v", 32.0: "s"}, palette="viridis")

sns.scatterplot(data=df, ax=hAx2, x='fp_th0p95', y='fn_th0p95', 
                hue='num_params', style='win_stride', markers={16.0: "^", 32.0: "d"}, palette="viridis")
hAx2.set_xlim([0,25])
hAx2.grid(True)
hAx2.set_title(f"Threshold fixed at 0.99, .95")
sns.move_legend(hAx2, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
plt.figure(figsize=(8,5))
hAx1 = plt.subplot(1,2,1)
sns.scatterplot(data=df, ax=hAx1, x='num_params', y='sparse1p0', hue='win_stride')
hAx1.grid(True)
hAx1.set_ylabel("Sparsity at 1% level")
sns.move_legend(hAx1, "upper left", bbox_to_anchor=(1, 1))


hAx2 = plt.subplot(1,2,2)
sns.scatterplot(data=df, ax=hAx2, x='num_params', y='false_detections', hue='win_stride')
hAx2.grid(True)
hAx2.set_ylabel("Sparsity at 1% level")
sns.move_legend(hAx2, "upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()

### L2 Experiment (old)

In [ ]:
# File path
file_path = "experiment_data/l2_summary_20dec2024.csv"
df = read_results_file(file_path)
# df["train_set"] = "sww_stride16.0_50clips_snr0.1_t1"
# df.insert(2, 'num_clips', value=50)
# df['num_params'][3:]=[44299, 44299, 44299, 119883, 119883, 119883, 376843, 376843, 376843, 522763, 522763, 522763]

df

In [ ]:
# hAx1 = plt.subplot(1,2,1)
hAx1= plt.gca()
sns.scatterplot(data=df, ax=hAx1, x='false_detections', y='false_rejections', hue='L2', palette=["red", "orange", "green", "blue"])
hAx1.grid(True)
sns.move_legend(hAx1, "upper left", bbox_to_anchor=(1, 1))


This dataset needs to have sparsity values extracted to be very informative.

### L2 Experiment 6 Jan 2025

In [ ]:
file_path = "experiment_data/l2_results_1.dat"
df = read_results_file(file_path)
df["train_set"] = "sww_stride16.0_50clips_snr0.1_t1"
# df["num_params"] = 522763
df.insert(2, 'num_clips', value=50)
df.insert(2, "l2_log", value=np.log10(df["l2"]+1e-9))


df

In [ ]:
# hAx1 = plt.subplot(1,2,1)
hAx1= plt.gca()
sns.scatterplot(data=df, ax=hAx1, x='fp_th0p99', y='fn_th0p99', hue='l2', 
                palette=["red", "orange", "green", "blue"],
                size="sparse1p0")
hAx1.grid(True)
sns.move_legend(hAx1, "upper left", bbox_to_anchor=(1, 1))


In [ ]:
hAx1= plt.gca()
sns.scatterplot(data=df, ax=hAx1, x='sparse0p1', y='l2_log')
hAx1.grid(True)
# sns.move_legend(hAx1, "upper left", bbox_to_anchor=(1, 1))

### L2 + Model-Size Experiment 6 Jan 2025

In [ ]:
!pwd

In [ ]:
file_path = "experiment_data/l2_model_results_1.dat"
df = read_results_file(file_path)
# df["train_set"] = "sww_stride16.0_50clips_snr0.1_t1"
# df["num_params"] = 522763
# df.insert(2, 'num_clips', value=50)
# df.insert(2, "l2_log", value=np.log10(df["l2"]+1e-9))


df

In [ ]:
hAx1= plt.gca()
sns.scatterplot(data=df, ax=hAx1, x='fp_th0p99', y='fn_th0p99', 
                # style='l2', markers={0.0: "o", 1e-5:"v", 1e-4:"^", 1e-3:"*"}, 
                # hue='num_params', palette="viridis",                
                hue='l2', palette=["red", "orange", "green", "blue"],
                size="num_params")
hAx1.grid(True)
sns.move_legend(hAx1, "upper left", bbox_to_anchor=(1, 1))